# Factor Research & Backtesting

This notebook demonstrates the core quant research workflow: building factor-return matrices, analyzing signal quality, and evaluating predictive power across horizons.

**What you'll learn:**
- Building backtest matrices with forward returns
- Factor-return correlation analysis
- Combining multiple feature sets
- Volatility regime analysis
- Signal decay across horizons
- Snapshot pinning for reproducibility

**Datasets:** BTCUSDT OHLCV (3,600 rows) with `candle_factors` and `risk_factors`

## 1. Pipeline Setup

In [ ]:
import subprocess
import os
from pathlib import Path

# Navigate to the project root (parent of notebooks/)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
elif not Path("config/datasets.yaml").exists():
    root = Path.cwd()
    while not (root / "config" / "datasets.yaml").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

# Initialize platform and run the full pipeline for both feature sets
subprocess.run(["adp", "init"], capture_output=True, check=True)
subprocess.run(["adp", "ingest", "ohlcv_btcusdt", "--force"], capture_output=True, check=True)
subprocess.run(["adp", "snapshot", "create", "ohlcv_btcusdt"], capture_output=True, check=True)
# candle_factors: SMAs, EWMA, rolling volatility, returns, VWAP
subprocess.run(["adp", "features", "build", "ohlcv_btcusdt", "candle_factors"], capture_output=True, check=True)
# risk_factors: realized volatility, z-score, rolling high/low
subprocess.run(["adp", "features", "build", "ohlcv_btcusdt", "risk_factors"], capture_output=True, check=True)
print("Pipeline complete: ohlcv_btcusdt -> candle_factors + risk_factors")

In [ ]:
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from adp import (
    load_features,
    build_backtest_matrix,
    list_snapshots,
    list_feature_sets,
)

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(50)

## 2. Building the Backtest Matrix

`build_backtest_matrix()` takes a feature set and computes forward returns at specified horizons. Forward returns are calculated as:

$$\text{fwd\_return}_N = \frac{\text{close}_{t+N}}{\text{close}_t} - 1$$

The last N rows have `null` forward returns (no future data), preventing look-ahead bias.

In [ ]:
# Build backtest matrix with multiple forward-return horizons
bt_lf = build_backtest_matrix(
    dataset="ohlcv_btcusdt",
    feature_set="candle_factors",
    forward_return_periods=[1, 5, 10, 60],
    price_column="close",
    sort_column="timestamp",
)

bt_df = bt_lf.collect()
print(f"Backtest matrix shape: {bt_df.shape}")
print(f"Columns: {bt_df.columns}")

# Preview: features + forward returns
print("\nPreview (rows 50-60):")
print(
    bt_df.select(
        ["timestamp", "close", "sma_10", "rolling_vol_5",
         "fwd_return_1", "fwd_return_5", "fwd_return_10", "fwd_return_60"]
    ).slice(50, 10)
)

## 3. Factor-Return Correlation Analysis

Compute Pearson correlations between each factor and forward returns at different horizons.

**Interpretation caveats:**

- **Non-stationary features** (`sma_10`, `sma_50`, `ewma_20`, `vwap`) are price-level series. Correlating non-stationary features with stationary returns can produce spurious correlations. For rigorous analysis, normalize these as ratios to close (e.g., `sma_10 / close - 1`) or use only stationary features (`rolling_vol_5`, `close_returns`, `close_log_returns`).
- **Overlapping returns** at horizons >1 share data across consecutive rows (e.g., `fwd_return_60` at time t and t+1 share 59 of 60 seconds). This inflates apparent statistical significance. The effective independent sample count is approximately `N / horizon`, not `N`.

In [ ]:
factors = ["rolling_vol_5", "sma_10", "sma_50", "ewma_20", "close_returns", "close_log_returns", "vwap"]
horizons = ["fwd_return_1", "fwd_return_5", "fwd_return_10", "fwd_return_60"]

# Drop nulls for clean correlation computation
clean = bt_df.drop_nulls(subset=factors + horizons)
print(f"Clean rows (no nulls): {clean.shape[0]} of {bt_df.shape[0]}")

# Build correlation table: Pearson corr of each factor vs each forward-return horizon
corr_data = []
for factor in factors:
    row = {"factor": factor}
    for horizon in horizons:
        corr = clean.select(pl.corr(factor, horizon)).item()
        row[horizon] = round(corr, 6)
    corr_data.append(row)

corr_table = pl.DataFrame(corr_data)
print("\n=== Factor-Return Correlations ===")
print(corr_table)

### Single-Horizon Correlation View

Bar chart of factor correlations with the 5-second forward return, for quick visual comparison.

In [ ]:
# Bar chart of factor-return correlations for fwd_return_5
corrs_5 = [
    clean.select(pl.corr(f, "fwd_return_5")).item()
    for f in factors
]

colors = ["steelblue" if c >= 0 else "salmon" for c in corrs_5]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(factors, corrs_5, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Pearson Correlation with fwd_return_5")
ax.set_title("Factor-Return Correlations (5-Second Forward Return)")
ax.axvline(x=0, color="gray", linewidth=0.5)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## 4. Combining Multiple Feature Sets

Load both `candle_factors` (trend signals) and `risk_factors` (risk metrics) and join them into a single research matrix.

In [ ]:
candle_df = load_features("ohlcv_btcusdt", "candle_factors").collect()
risk_df = load_features("ohlcv_btcusdt", "risk_factors").collect()

print(f"Candle factors: {candle_df.shape} — {candle_df.columns}")
print(f"Risk factors:   {risk_df.shape} — {risk_df.columns}")

# Inner join on timestamp — both originate from the same snapshot, so no rows are lost
combined = candle_df.join(
    risk_df.select(
        ["timestamp", "realized_vol_20", "close_zscore_30", "rolling_low_10", "rolling_high_10"]
    ),
    on="timestamp",
    how="inner",
)
print(f"\nCombined matrix: {combined.shape}")
print(combined.select(
    ["timestamp", "close", "sma_10", "rolling_vol_5",
     "realized_vol_20", "close_zscore_30", "rolling_low_10", "rolling_high_10"]
).slice(50, 10))

### Volatility Measure Comparison

Overlay `rolling_vol_5` (candle_factors) and `realized_vol_20` (risk_factors) to visualize how the short-window and long-window volatility measures track each other.

In [ ]:
# Time series overlay of both volatility measures
vol_df = combined.drop_nulls(subset=["rolling_vol_5", "realized_vol_20"]).sort("timestamp")
ts = vol_df["timestamp"].to_list()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ts, vol_df["rolling_vol_5"].to_list(), label="rolling_vol_5 (candle)", linewidth=1.2)
ax.plot(ts, vol_df["realized_vol_20"].to_list(), label="realized_vol_20 (risk)", linewidth=1.2)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Volatility")
ax.set_title("Volatility Comparison: Rolling Vol(5) vs Realized Vol(20)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Volatility Regime Analysis

Split the data into high-volatility and low-volatility regimes based on the median of `realized_vol_20`, then compare forward returns in each regime.

In [ ]:
# Compute forward returns on the combined matrix (same logic as build_backtest_matrix)
combined_sorted = combined.sort("timestamp")
for period in [1, 5, 10, 60]:
    combined_sorted = combined_sorted.with_columns(
        pl.when(pl.col("close").abs() > 1e-15)
        .then(pl.col("close").shift(-period) / pl.col("close") - 1.0)
        .otherwise(None)
        .alias(f"fwd_return_{period}")
    )

# Split into high/low volatility regimes using realized_vol_20 median as threshold
vol_clean = combined_sorted.drop_nulls(subset=["realized_vol_20", "fwd_return_5"])
vol_median = vol_clean["realized_vol_20"].median()
print(f"Realized vol(20) median: {vol_median:.6f}")

vol_clean = vol_clean.with_columns(
    pl.when(pl.col("realized_vol_20") > vol_median)
    .then(pl.lit("High Vol"))
    .otherwise(pl.lit("Low Vol"))
    .alias("vol_regime")
)

# Compare forward returns by regime
# Note: overlapping returns reduce effective sample size (see Section 3 caveat)
regime_stats = (
    vol_clean
    .group_by("vol_regime")
    .agg([
        pl.col("fwd_return_1").mean().round(8).alias("avg_fwd_1"),
        pl.col("fwd_return_5").mean().round(8).alias("avg_fwd_5"),
        pl.col("fwd_return_10").mean().round(8).alias("avg_fwd_10"),
        pl.col("fwd_return_5").std().round(8).alias("std_fwd_5"),
        pl.col("fwd_return_5").count().alias("count"),
    ])
)

print("\n=== Forward Returns by Volatility Regime ===")
print(regime_stats)

## 6. Signal Decay Analysis

Plot how factor-return correlations change across forward-return horizons. Factors whose correlation decays quickly are short-lived signals; persistent correlations suggest structural relationships. Raw (signed) correlations are shown so that sign reversals (momentum vs mean-reversion) are visible.

In [ ]:
analysis_factors = ["rolling_vol_5", "sma_10", "close_returns", "close_zscore_30"]
horizon_periods = [1, 5, 10, 60]

# Use combined_sorted which has both feature sets + forward returns
# Note: close_zscore_30 has 29-row warmup; combined with fwd_return_60 tail nulls,
# effective sample is reduced by ~89 rows from 3,600 total
decay_clean = combined_sorted.drop_nulls(
    subset=analysis_factors + [f"fwd_return_{p}" for p in horizon_periods]
)

fig, ax = plt.subplots(figsize=(10, 6))

# Plot signed correlation at each horizon to reveal direction changes (momentum vs mean-reversion)
for factor in analysis_factors:
    corrs = []
    for period in horizon_periods:
        corr = decay_clean.select(pl.corr(factor, f"fwd_return_{period}")).item()
        corrs.append(corr)  # raw (signed) correlation to show direction changes
    ax.plot(horizon_periods, corrs, marker="o", label=factor, linewidth=2)

ax.set_xlabel("Forward Return Horizon (seconds)")
ax.set_ylabel("Correlation")
ax.set_title("Signal Decay: Factor-Return Correlation by Horizon")
ax.axhline(y=0, color="gray", linewidth=0.5, linestyle="--")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Factor Distribution Analysis

Examine the statistical distributions of key factors to understand their behavior.

In [ ]:
dist_factors = ["rolling_vol_5", "close_returns", "close_zscore_30"]
titles = ["Rolling Volatility (5)", "Close Returns", "Z-Score (30)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, factor, title in zip(axes, dist_factors, titles):
    data = combined_sorted[factor].drop_nulls().to_list()
    ax.hist(data, bins=50, alpha=0.7, edgecolor="black", linewidth=0.5)
    ax.set_title(title)
    ax.set_xlabel(factor)
    ax.set_ylabel("Frequency")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Snapshot Pinning for Reproducibility

To ensure your research is reproducible, pin the exact snapshot IDs used in your analysis. This guarantees identical results regardless of newer data ingestions.

In [ ]:
# Record the snapshot IDs used in this analysis
print("=== Reproducibility Record ===")
print("\nDataset snapshots:")
print(list_snapshots("ohlcv_btcusdt"))

print("\nFeature sets:")
print(list_feature_sets("ohlcv_btcusdt"))

print("\nTo reproduce this analysis, pin the snapshot IDs:")
print('  lf = load_features("ohlcv_btcusdt", "candle_factors", snapshot_id="<ID>")')